# 12_09_検索側と生成側を分けて測る

コード：`ch12/blocks/12_09_検索側と生成側を分けて測る.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2', 'transformers==5.22.1', 'tiktoken==0.14.0', 'scikit-learn>=1.5,<2', 'pydantic==2.14.2', 'outlines==1.3.3', 'lancedb==0.41.0', 'pyarrow==26.0.1', 'sentence-transformers==6.2.0']], check=True)


## 本文の掲載コード（参照）


```python
import numpy as np

def retrieval_metrics(dataset, retrieve, k=10):
    """dataset: [{"q": str, "gold": set[doc_id]}, ...]"""
    rec, mrr, ndcg = [], [], []
    for ex in dataset:
        ids  = [d["id"] for d in retrieve(ex["q"], top_k=k)]
        gold = ex["gold"]
        if not gold:
            raise ValueError("正解集合が空の質問は、棄権を測る別の評価群にする")
        hit  = [1 if i in gold else 0 for i in ids]

        rec.append(len(gold & set(ids)) / max(1, len(gold)))          # Recall@k
        first = next((r for r, h in enumerate(hit) if h), None)
        mrr.append(0.0 if first is None else 1 / (first + 1))          # MRR
        dcg   = sum(h / np.log2(r + 2) for r, h in enumerate(hit))
        idcg  = sum(1 / np.log2(r + 2) for r in range(min(len(gold), k)))
        ndcg.append(dcg / idcg if idcg else 0.0)                       # nDCG@k
    return {"recall@k": np.mean(rec), "mrr": np.mean(mrr), "ndcg@k": np.mean(ndcg)}
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "12", "--name=rag", "--skip-docker"], cwd=CODE, check=True)
